# Section 1 - Intro to NLP and LLMs

By the end of this workshop you will have built a **RAG system**: a search engine over every NUS module, connected to an LLM that answers questions using what the search finds. This section lays the groundwork in three parts:

1. **Setup:** load the module data and connect your own Gemini API key.
2. **From text to numbers:** why text is hard for computers, what tokens are, and how an LLM actually writes.
3. **Where LLMs break:** ask Gemini about NUS modules, check its answers against the real data, and see why RAG is the fix.


## Setup

Run the two cells in this part now, while everyone is arriving.

On Colab, the first cell starts by downloading this week's folder from GitHub. Then it loads three things through `rag_utils.py`, a small helper file that all three sections share:

- `modules`: every module NUS offers in AY 2026-2027, taken from [NUSMods](https://nusmods.com). Each module is a Python dictionary with its code, title, description, workload and more. The organisers downloaded this in advance and saved it in `data/`, so the workshop does not depend on NUSMods being up.
- `embeddings`: one row of numbers per module. You will meet these in Section 2.
- `model`: `all-MiniLM-L6-v2`, a small model that turns text into numbers. It runs on your own CPU and needs no API key. The first run downloads it (about 90 MB).

In [ ]:
# %pip install -q google-genai sentence-transformers numpy matplotlib
import json
import os
import re
import sys
import textwrap
import urllib.request
import warnings

# On Colab: download only the week-11-nlp folder of the workshop repo (a few seconds)
if "google.colab" in sys.modules:
    if not os.path.exists("/content/Sem-1"):
        !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/NUS-SDS-Workshops-AY2627/Sem-1.git /content/Sem-1
        !git -C /content/Sem-1 sparse-checkout set week-11-nlp
    %cd /content/Sem-1/week-11-nlp

sys.path.append("..")  # rag_utils.py is one folder up from notebooks/
os.environ["TRANSFORMERS_VERBOSITY"] = os.environ["HF_HUB_VERBOSITY"] = "error"  # hide harmless warnings
warnings.filterwarnings("ignore", message=".*HF_TOKEN.*")

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

import rag_utils
from rag_utils import build_chunk, keyword_search, search

modules, embeddings, model = rag_utils.load()
by_code = {module["moduleCode"]: module for module in modules}  # "CS4248" -> its record

print(f"{len(modules)} modules offered in AY {rag_utils.AY}, fetched from NUSMods on {rag_utils.FETCH_DATE}")

### Connect Gemini

Each of you needs your **own** free API key: the free quota is counted per Google Cloud project, so 40 people sharing one key would run out within minutes.

> **Your Gemini API key (one-time, about 2 minutes)**
> 1. Go to **[aistudio.google.com/apikey](https://aistudio.google.com/apikey)**, sign in with a *personal* Google account, find the 🔑 icon, click **Create API key** and copy the key. It's free — **don't** set up billing, because that turns the free tier off for the project.
> 2. In Colab, click the **key icon 🔑 (Secrets)** in the left sidebar → **Add new secret**.
> 3. Name: `GEMINI_API_KEY` · Value: paste your key · switch **Notebook access** on.
>
> No secret set up? The next cell asks you to paste the key instead. Never type your key into the code itself: it would be saved inside the notebook, and anyone you share the notebook with could use up your quota.

The cell below connects `ask_gemini` (from `rag_utils.py`), the one function we use to talk to Gemini. It sends your prompt and nothing else: no extra documents, and no Google Search. If the call fails, for example because the free quota has run out, it prints the reason and returns a saved answer instead of crashing.

In [ ]:
from rag_utils import ask_gemini

# Looks for your key in Colab Secrets first, then asks you to paste it.
rag_utils.connect_gemini()

print("Gemini says:", ask_gemini("Reply with exactly: ready"))

You should see `Gemini says: ready`. If you see `Gemini call failed` instead, the message tells you why. The most common one is `API key not valid`: copy the key again, update the secret, and re-run the cell. If it still fails, raise your hand. You can follow the Gemini demos on the main screen in the meantime.

## 1.1 Text is hard for computers

Computers only work with numbers. Text is not.

Even turning text into numbers is not enough, because **the meaning of a sentence is not in its characters**. Two strings that mean the same thing can look completely different, and one string can mean two different things.

In [ ]:
print("car" == "automobile")  # same meaning, different characters
print("bank" == "bank")       # same characters... river bank or money bank?

# Text CAN be turned into numbers, one per character
print([ord(character) for character in "car"])
print([ord(character) for character in "automobile"])

## 1.2 Ctrl+F fails

 `keyword_search` does the same across every module description: it returns the modules whose description contains your text, character for character.

Say you want a module about how computers understand language.

In [ ]:
def print_matches(query, k=5):
    """Run a keyword search and print the first k matches."""
    matches = keyword_search(query)
    print(f'"{query}": {len(matches)} matches')
    for match in matches[:k]:
        print(f"    {match['moduleCode']:<9} {match['title']}")


print_matches("how computers understand language")

In [ ]:
print_matches("natural language processing")
print_matches("language")

Keyword search compares characters, and meaning is not in the characters. To find anything, you have to guess the exact words the author used.

## 1.3 Tokens

Language models split text into **tokens**: common words stay whole, and rare words are chopped into smaller pieces that the model has seen before.

Every model comes with a **tokenizer** that does this splitting. Here is the tokenizer of `all-MiniLM-L6-v2` reading a real module description.

In [ ]:
module = by_code["CS4248"]
text = module["description"]

print(module["moduleCode"], "-", module["title"])
print(textwrap.fill(text, 100))
print()

tokens = model.tokenizer.tokenize(text)
print(f"{len(text.split())} words -> {len(tokens)} tokens")
print(tokens)

**YOUR TURN:** add your own words to the list and re-run. Try your name, Singlish, a long made-up word, or another language.

In [ ]:
# YOUR TURN: add your own words, then re-run
my_words = ["language", "tokenization", "NUSMods", "Singapore", "supercalifragilistic", "数据科学"]

for word in my_words:
    print(f"{word:>22} -> {model.tokenizer.tokenize(word)}")

# Each token is really an ID number in the model's vocabulary
print()
print("Token IDs:", model.tokenizer.encode("natural language processing", add_special_tokens=False))

`[UNK]` means **unknown**: the character is not in the model's vocabulary at all. 
The IDs on the last line are what the model actually receives. 

### The token budget

Every model can only read a limited number of tokens at once. 

In [ ]:
limit = model.max_seq_length
description_lengths = [len(model.tokenizer.tokenize(m["description"])) for m in modules]
longest = int(np.argmax(description_lengths))

print(f"MiniLM reads at most {limit} tokens")
print(f"Longest description: {description_lengths[longest]} tokens ({modules[longest]['moduleCode']})")
print(f"Descriptions over the limit: {sum(n > limit for n in description_lengths)} of {len(modules)}")

# Gemini uses a different tokenizer, so the same text gives a different count
try:
    gemini_count = rag_utils.gemini_client().models.count_tokens(model=rag_utils.GEMINI_MODEL, contents=text).total_tokens
    print(f"\nThe CS4248 description: MiniLM = {len(tokens)} tokens, Gemini = {gemini_count} tokens")
except Exception:
    pass

## 1.4 How an LLM works

An **LLM** (large language model)  does one thing: given the text so far, it predicts **which token comes next**. 

To watch it happen, we load a tiny LLM called **SmolLM2**. It has 135 million parameters, small enough to run on a laptop CPU (about a 270 MB download). Gemini works the same way, just thousands of times bigger.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

LM_NAME = "HuggingFaceTB/SmolLM2-135M"
lm_tokenizer = AutoTokenizer.from_pretrained(LM_NAME)
lm = AutoModelForCausalLM.from_pretrained(LM_NAME)
lm.eval()


def show_next_tokens(prompt, k=5):
    """Show the model's top k guesses for the very next token."""
    ids = lm_tokenizer(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        probabilities = torch.softmax(lm(ids).logits[0, -1], dim=-1)
    top = torch.topk(probabilities, k)
    print(f"{prompt!r} ...")
    for probability, token_id in zip(top.values, top.indices):
        token = lm_tokenizer.decode([int(token_id)])
        print(f"   {token!r:<16} {probability.item():6.1%}  " + "#" * int(probability.item() * 40))


def generate_step_by_step(prompt, n=15):
    """Generate n tokens one at a time, always picking the most likely one."""
    ids = lm_tokenizer(prompt, return_tensors="pt").input_ids
    for step in range(n):
        with torch.no_grad():
            next_id = int(lm(ids).logits[0, -1].argmax())
        ids = torch.cat([ids, torch.tensor([[next_id]])], dim=1)
        print(f"step {step + 1:>2}: + {lm_tokenizer.decode([next_id])!r}")
    print()
    print("Full text:", lm_tokenizer.decode(ids[0], skip_special_tokens=True))


show_next_tokens("The National University of Singapore is located in")

Now let the model keep going. At every step it takes the most likely token and adds it to the text.

In [ ]:
generate_step_by_step("The National University of Singapore is located in", n=12)

It reads smoothly. But look closely: Singapore is not "a city in the southern part of the country". It **is** the country. The model aims for text that sounds natural, and that is not the same as text that is true.

What happens when we ask about something the model cannot possibly know?

In [ ]:
show_next_tokens("The NUS module CS4248 is titled")
generate_step_by_step("The NUS module CS4248 is titled", n=15)

print()
print("Real title:", by_code["CS4248"]["title"])

## 1.5 Where LLMs break

SmolLM2 is tiny. Does a real model do better? We will ask Gemini about NUS modules, giving it **no extra information**, and then **check its answer** against the real NUSMods data.

`verify` does the checking. It finds every module code in an answer, such as `CS4248`, and looks it up in this year's module list.

In [ ]:
MODULE_CODE = re.compile(r"\b[A-Z]{2,4}\d{4}[A-Z]{0,2}\b")  # e.g. CS4248, CS2103T


def verify(answer):
    """Look up every module code in the answer in the real NUSMods data."""
    codes = list(dict.fromkeys(MODULE_CODE.findall(answer)))  # unique, in order
    for code in codes:
        real = by_code.get(code)
        if real:
            print(f"  ✅ {code:<9} real title: {real['title']}")
        else:
            print(f"  ❌ {code:<9} not offered in AY {rag_utils.AY}, or does not exist")
        said = next(line for line in answer.splitlines() if code in line)  # where Gemini mentioned it
        print(f"{'':15}Gemini wrote: {textwrap.shorten(re.sub(r'[*_#`|>]', '', said), 90)}")
    print()
    print(f"{sum(code in by_code for code in codes)} of {len(codes)} module codes are real.")

### Ask, then verify

First, the question. Gemini sees only this sentence.

In [ ]:
question = "Which NUS modules cover attention mechanisms? List the module codes and titles."

answer = ask_gemini(question)
display(Markdown(answer))

We verify the answer in the following block. 

In [ ]:
verify(answer)

**How to read it:**

- ❌ means the code is not in this year's NUSMods data. Either the module does not exist, or it is not running this year.
- ✅ with a different title means the code is real, but Gemini attached the wrong module to it, or used an old name.
- ✅ with a matching title means Gemini got that one right. You only know because you checked.

### Numbers it could easily invent

Specific numbers are the easiest thing for an LLM to get plausibly wrong. NUSMods lists the weekly workload of every module in hours.

In [ ]:
module_code = "CS4248"

print("GEMINI SAYS:")
print(ask_gemini(f"What is the weekly workload of NUS module {module_code} in hours, "
                 "as lecture / tutorial / lab / project / preparation?"))
print()
print("NUSMODS SAYS:", rag_utils.format_workload(by_code.get(module_code, {}).get("workload")) or "(not listed)")

Hours per week that look reasonable are exactly what next-token prediction produces, whether or not they are true. Try a module you have taken, such as `CS2040S` or `ST2334`, and re-run.

### Right last year, wrong this year

**Context:** CS4248 was called *Natural Language Processing* for years. In AY 2026-2027 it was renamed *Introduction to Large Language Models*. 

In [ ]:
module_code = "CS4248"
print("GEMINI SAYS:", ask_gemini(f"What is the official title of the NUS module {module_code}? Reply with the title only."))

with urllib.request.urlopen("https://api.nusmods.com/v2/2025-2026/moduleList.json") as response:
    last_year = {m["moduleCode"]: m["title"] for m in json.load(response)}

print("NUSMODS AY 2025-2026:", last_year.get(module_code))
print("NUSMODS AY 2026-2027:", by_code.get(module_code, {}).get("title"))

**YOUR TURN:** ask Gemini your own question about NUS modules, then check it. Ask for module codes, so that `verify` has something to check. Some things to try:

- A topic you care about: *"Which NUS modules teach reinforcement learning?"*
- Something new: *"Which NUS modules teach agentic AI?"*
- A module you have taken: *"What is CS2040S about, and what modules come after it?"*

In [ ]:
# YOUR TURN: ask your own question, then re-run
my_question = "Which NUS modules should I take to learn about recommender systems? List the module codes and titles."

my_answer = ask_gemini(my_question)
display(Markdown(my_answer))
print("-" * 60)
verify(my_answer)

## 1.6 The RAG idea

The following block counts below use MiniLM's tokenizer, that how many tokens it would take to paste all NUSMods info into a prompt. Treat them as estimates.

 `build_chunk` turns one module into the text we would paste. 

In [ ]:
chunks = [build_chunk(m) for m in modules]
chunk_lengths = [len(ids) for ids in model.tokenizer(chunks, verbose=False)["input_ids"]]

print(f"All {len(modules)} modules: about {sum(chunk_lengths):,} tokens, in every single prompt")
print(f"5 relevant modules: about {sum(chunk_lengths[:5]):,} tokens")

try:
    gemini_limit = rag_utils.gemini_client().models.get(model=rag_utils.GEMINI_MODEL).input_token_limit
    print(f"{rag_utils.GEMINI_MODEL} reads at most {gemini_limit:,} tokens per prompt")
except Exception:
    pass


**So fetch only the few modules that matter, and let the LLM answer from those.** That is **RAG**, retrieval-augmented generation:

```
 ┌─────────────┐     ┌─────────────┐     ┌────────────────────┐     ┌───────────────┐
 │ 1. Question │ ──> │ 2. Retrieve │ ──> │ 3. Augment         │ ──> │ 4. Generate   │
 │  from user  │     │  top-k most │     │  question + those  │     │  LLM answers  │
 │             │     │  relevant   │     │  modules -> prompt │     │  FROM them,   │
 │             │     │  modules    │     │                    │     │  with sources │
 └─────────────┘     └─────────────┘     └────────────────────┘     └───────────────┘
                       Section 2           Section 3                  Section 3
```


### If you're stuck

Run the cell below. It reloads everything Section 2 needs.

In [ ]:
# Reloads the data and the embedding model if anything above failed
import sys

sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np

import rag_utils
from rag_utils import ask_gemini, build_chunk, keyword_search, search

modules, embeddings, model = rag_utils.load()
by_code = {module["moduleCode"]: module for module in modules}
print(len(modules), "modules, embeddings", embeddings.shape)